# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Fresh CPU validation failed: the BrAPI Test Server (test-server.brapi.org), the backend of the author's bundled config_demo.ini, returned persistent nginx 502 for the whole window (Colab probes 22:53-23:01 UTC 2026-10-24, incl. 6 retries at 30 s). brapi_proxy --demo reported 'server test1 unreachable' (cell 5 RuntimeError), downstream cells 404-cascaded. Third-party outage, not a Colab/CLI/code defect; historical 2026-10-06 run succeeded against the same endpoint. Notebook (with one fix: removed matplotlib.use('Agg') so the result graph saves an inline image/png preview) is statically valid and preserved in deliverables/ for the unverified-reference path. Retry when the test server recovers.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Integrating plant phenotypic and genotypic data in the AGENT project: a BrAPI service implementation.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-d0270e5cfded603928047f692df3bfce)


# AGENT BrAPI proxy — demo-mode merge and paging verification

**Paper:** "Integrating plant phenotypic and genotypic data in the AGENT project: a BrAPI service implementation."
*Bioinformatics* 42(5):btag287 (2026). DOI: [10.1093/bioinformatics/btag287](https://doi.org/10.1093/bioinformatics/btag287) — PMC13211978.

**Author code:** [`matthijsbrouwer/brapi-proxy`](https://github.com/matthijsbrouwer/brapi-proxy) @ commit `6355e8197d1a8e8d1c2e593f2b51f3bf7bbfa6f6` (v0.9.0, MIT; pin established via the paper's Zenodo record DOI 10.5281/zenodo.19436445). PyPI: [`brapi-proxy`](https://pypi.org/project/brapi-proxy/).

**Scope (partial demonstration):** start the author's `brapi_proxy` service in its **demo mode** (an embedded configuration based on the public [BrAPI Test Server](https://test-server.brapi.org/brapi/v2/)) and verify the paper's §2.2/§3 proxy behaviour: one unified BrAPI 2.1 service that merges list-structured responses from **multiple underlying endpoints** with **recalculated paging**, including crop-prefixed identifiers. This is a one-service network/JSON check, **not** a reproduction of the paper's marker–trait validation (§3.1), which requires the unpublished AGENT/DivBrowse live endpoints.

**実行環境（日本語）:** CPU のみで動作します（純 Python の HTTP プロキシ＋公開 BrAPI テストサーバーへの要求）。GPU は不要です。所要時間は数分、追加の重み付け・データセットのダウンロードはありません。

**Execution status:** not yet executed — run all cells top-to-bottom on a fresh CPU runtime.

## Runtime selection

Select **Runtime → Change runtime type → CPU** (the default free CPU runtime is sufficient; no GPU/TPU needed).
Then **Runtime → Run all**. Expected duration: ~2–4 minutes including `pip install`.
**実行手順（日本語）:** 「ランタイム → ランタイムのタイプを変更 → CPU」を選び、「ランタイム → すべてのセルを実行」してください。GPU は不要で、数分で完了します。

### Install the pinned author package

Installs `brapi_proxy==0.9.0` from PyPI (the same version pinned by the paper's Zenodo record). This brings the author's Flask/waitress service and its bundled `config_demo.ini`.
**説明（日本語）:** 論文の Zenodo レコードが固定するバージョン 0.9.0 を PyPI からインストールします。デモ用設定ファイルはパッケージ内に同梱されています。

In [ ]:
%pip install -q brapi_proxy==0.9.0

import brapi_proxy
from brapi_proxy._version import __version__
assert __version__ == "0.9.0", f"unexpected brapi_proxy version {__version__}"
print("brapi_proxy", __version__)

### Launch the demo service

Starts `brapi_proxy --demo` as a background process (the author's documented demo entry point: a service on port 8080 configured from the embedded `config_demo.ini`, based on the public BrAPI Test Server). Server output is streamed to a log file so failures stay visible; we poll until the port answers.
**説明（日本語）:** 作者が文書化した `--demo` モードでサービスをバックグラウンド起動し、ポート 8080 が応答するまで待ちます。ログはファイルに残し、失敗を見える化します。

In [ ]:
import subprocess, time, socket, pathlib, requests

LOG = pathlib.Path("/content/brapi_proxy_demo.log")
proc = subprocess.Popen(["brapi_proxy", "--demo"],
                        stdout=open(LOG, "ab"), stderr=subprocess.STDOUT)
print("started pid", proc.pid)

BASE = "http://localhost:8080/brapi/v2"
for attempt in range(60):          # bounded wait: ~60 s max
    try:
        r = requests.get(f"{BASE}/serverinfo", timeout=2)
        if r.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(f"demo service did not come up; log:\n{LOG.read_text()[:2000]}")
print("service is up:", r.json())

### Merged response check 1: `commoncropnames`

The demo configuration defines several underlying `server.*` sections against the BrAPI Test Server. `/commoncropnames` is a list endpoint served by more than one of them, so the proxy must return a single merged `data` list (paper §2.2: merge list-structured responses).
**説明（日本語）:** 複数のバックエンドが担うリスト系エンドポイントが、単一の統合レスポンスとして返ることを確認します。

In [ ]:
r = requests.get(f"{BASE}/commoncropnames", timeout=30)
r.raise_for_status()
crops = r.json()
print("metadata:", crops.get("metadata"))
print("data:", crops.get("data"))
assert isinstance(crops.get("data"), list) and len(crops["data"]) >= 1

### Merged response check 2: prefixed `variants` and `callsets`

The README documents per-server identifier prefixes (e.g. `barley:`, `wheat:`) that keep identifiers unique across crop-specific backends (paper §3: "prefix crop-specific identifiers ... to keep identifiers unique across sources"). `/variants` and `/callsets` should return records whose names/IDs carry these prefixes, merged from the two prefixed servers.
**説明（日本語）:** `variants` / `callsets` の統合結果に `barley:` / `wheat:` など接頭辞付き ID が現れることを確認します。

In [ ]:
import collections

def get(path, **params):
    r = requests.get(f"{BASE}{path}", params=params, timeout=60)
    r.raise_for_status()
    return r.json()

variants = get("/variants", pageSize=50)
callsets = get("/callsets", pageSize=50)
print("variants metadata:", variants["metadata"])
print("callsets metadata:", callsets["metadata"])

vnames = [v.get("variantSetId") or v.get("name") for v in variants["data"]]
print("first variant records:", [ (v.get('variantDbId'), v.get('name')) for v in variants['data'][:5] ])
print("first callset records:", [ (c.get('callSetDbId'), c.get('callSetName')) for c in callsets['data'][:5] ])

prefixed = [str(c.get("callSetName", "")) for c in callsets["data"]]
prefix_counts = collections.Counter(n.split(":")[0] for n in prefixed if ":" in n)
print("callset name prefixes:", dict(prefix_counts))

### Paging check: recalculated pagination across backends

Paper §2.2 / Fig. 2: when list responses are merged, the proxy recalculates paging metadata (`pageSize`, `totalCount`) and serves consistent pages. We request page 0 and page 1 of `/variants` and verify disjoint record sets with stable metadata.
**説明（日本語）:** 統合後に `metadata.pagination` が再計算され、ページ 0 と 1 が重複なく取得できることを検証します。

In [ ]:
p0 = get("/variants", page=0, pageSize=10)
p1 = get("/variants", page=1, pageSize=10)
pag0 = p0["metadata"].get("pagination", {})
pag1 = p1["metadata"].get("pagination", {})
print("page0 pagination:", pag0)
print("page1 pagination:", pag1)

ids0 = [v.get("variantDbId") for v in p0["data"]]
ids1 = [v.get("variantDbId") for v in p1["data"]]
assert not set(ids0) & set(ids1), "page overlap detected"
assert pag0.get("totalCount") == pag1.get("totalCount"), "totalCount changed across pages"
print(f"OK: {len(ids0)} + {len(ids1)} disjoint records, totalCount={pag0.get('totalCount')}")

### Result summary and graph

Collect the record counts observed through the proxy for a few endpoints and plot them. This graph is **created for this notebook** from the live demo service (not a paper figure).
**説明（日本語）:** デモサービス経由で観測した各エンドポイントのレコード数を表とグラフにまとめます。このグラフは本ノートブックで作成した追加の可視化であり、論文の図ではありません。

In [ ]:
import matplotlib.pyplot as plt

endpoints = {}
for path in ["/commoncropnames", "/variants", "/callsets", "/studies"]:
    try:
        js = get(path, pageSize=1000)
        endpoints[path] = len(js.get("data", []))
    except Exception as e:
        endpoints[path] = 0
        print(path, "->", type(e).__name__, e)

import pandas as pd
summary = pd.DataFrame(sorted(endpoints.items()), columns=["endpoint", "records_via_proxy"])
print(summary.to_string(index=False))
summary.to_csv("/content/brapi_proxy_demo_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary["endpoint"], summary["records_via_proxy"], color="#4C72B0")
ax.set_ylabel("records returned via proxy (pageSize=1000)")
ax.set_title("brapi_proxy demo: records merged from underlying BrAPI test servers\n(additional visualization for this notebook, not a paper figure)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("/content/brapi_proxy_demo_summary.png", dpi=120)
plt.show()

## Interpretation and limitations

- What was verified here (if all cells above passed): the author's `brapi_proxy` 0.9.0 **demo mode** starts from the embedded config, merges list responses from multiple underlying BrAPI endpoints into one service, exposes per-server prefixed identifiers, and recalculates pagination metadata — the proxy behaviour described in paper §2.2 and §3 (Figs. 2–4).
- **Not reproduced:** the §3.1 marker–trait validation pipeline (barley plant-height marker SCRI_RS_154574 at 6H:17009885/17007008). It requires the AGENT/DivBrowse production endpoints and the AGENT database, whose deployment URLs are not published in the paper (see paper §3.1 and the PhenoPaper investigation blockers). The supplementary notebook (PMC13211978, `btag287_supplementary_data.zip`) was also not exercised.
- Results depend on the live public BrAPI Test Server contents at execution time; record counts may vary between runs.
- 検証内容の要約（日本語）: デモモードで作者のプロキシが複数バックエンドのレスポンスを統合し、接頭辞付き ID と再計算されたページングを提供することを確認しました。論文 §3.1 のマーカー・形質検証パイプラインは、公開されていない AGENT エンドポイントが必要なため対象外です。

**References**
1. van Treuren R. et al., *Bioinformatics* 42(5):btag287 (2026). (Author list as per the published record, PMC13211978.)
2. `matthijsbrouwer/brapi-proxy` @ `6355e8197d1a8e8d1c2e593f2b51f3bf7bbfa6f6`, MIT license — README demo instructions and `src/brapi_proxy/{service,brapi}.py`.
3. [BrAPI Test Server](https://test-server.brapi.org/brapi/v2/) — underlying public service used by the demo config.